<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_03_gnn_six_bus_network.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 03 — Node Regression on a Six-Bus Network

**Deep Learning for Engineering · Aalborg University · Part 1**

This is the notebook the exercise set is built around.

The task is *node regression*: given what is specified at every bus, predict
the electrical state at every bus — the voltage magnitude |V| and angle θ that a
power-flow calculation would give. Six input vectors in, six output vectors out,
on a fixed network of eight lines.

What happens here.

1. You meet the power-flow problem and the dataset: 800 operating points, each
   solved with a full AC power flow.
2. You build a message-passing layer and train a small graph network on it.
3. You time it against the AC power flow it learned from, on this grid and
   on larger ones.
4. You compare it with a dense network, renumber the buses, vary the depth,
   and trip a line, and see which model survives each.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_5_core as core
core.keep_outputs()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_5_core as core

np.set_printoptions(precision=4, suppress=True)

# seed 0: the same "random" numbers on every machine, so the initial weights,
# the training run and every printed number below are the same for you as in
# the "What you should see" notes. A different number means different code.
core.set_seed(0)

data = core.six_bus_dataset(n_cases=800, seed=12)
X, Y, A, E = data["X"], data["Y"], data["A"], data["E"]

print("X, node features :", X.shape, "  (cases, buses, numbers given per bus)")
print("   the numbers   :", data["feature_names"].tolist())
print("Y, node targets  :", Y.shape, "  (cases, buses, numbers solved per bus)")
print("   the numbers   :", data["target_names"].tolist())
print("A, adjacency     :", A.shape, "     (buses, buses)")
print("E, edge features :", E.shape, "     (lines, [R, X])")

**What you should see.**

```
X, node features : (800, 6, 6)   (cases, buses, numbers given per bus)
   the numbers   : ['P', 'Q', 'V_set', 'is_slack', 'is_PV', 'is_PQ']
Y, node targets  : (800, 6, 2)   (cases, buses, numbers solved per bus)
   the numbers   : ['theta [rad]', '|V| [p.u.]']
A, adjacency     : (6, 6)      (buses, buses)
E, edge features : (8, 2)      (lines, [R, X])
```

**Reading a shape.** Each number in brackets is the length of one axis of the
array. `(800, 6, 6)` is 800 operating points, times 6 buses, times 6 numbers per
bus. `(800, 6, 2)` is the same 800 cases and 6 buses with 2 answers per bus, the
angle and the voltage magnitude. The buses are kept on an axis of their own —
not flattened into one long row of 36 — so that the model can be built to treat
every bus by the same rule.

**Why `set_seed(0)`.** Training starts from random weights. Fixing the seed
makes those random numbers the same on every machine, so your run reproduces the
numbers printed in these notes, and a different number means a difference in
your code. Zero is not special; any fixed number would do.

---

## 1 · The power-flow problem, and the 800 cases

A **power flow** (load flow) finds the voltage at every bus — magnitude |V| and
angle θ — given what is connected there. Each bus has four quantities, P, Q, |V|
and θ. Two are specified and two are solved, and which two depends on the bus
**type**:

| type | buses here | given | solved |
| --- | --- | --- | --- |
| slack (reference) | 0, the large generator | \|V\| = 1.03 p.u. and θ = 0 | P and Q |
| PV (generator) | 1 | P and \|V\| = 1.02 p.u. | Q and θ |
| PQ (load, HVDC converter) | 2, 3, 4 and 5 | P and Q | \|V\| and θ |

The slack bus is needed because the losses in the lines are not known until the
flow is solved: one generator has to supply whatever the others do not. It also
holds the angle reference, θ = 0 — only angle *differences* drive power. A
generator holds its voltage magnitude with its excitation, so at buses 0 and 1
|V| does not change from case to case.

**The 800 cases.** Each case is one operating point of the same network. The
network, its impedances and the two voltage set-points never change. What is
drawn at random, independently and uniformly, is

* the PV generator's output at bus 1: P from 0.3 to 1.0 p.u. (30 to 100 MW);
* each load at buses 2, 3 and 4: P from 0.5 to 1.2 p.u. and Q from 0.1 to 0.4 p.u.;
* the HVDC infeed at bus 5: P from 0.1 to 0.5 p.u. and Q from −0.1 to 0.1 p.u.

For each draw the full AC power flow is solved by Newton-Raphson
(`core.ac_power_flow`), which gives |V| and θ at every bus and the P and Q of
the slack bus and the Q of the PV bus. All quantities are in **per unit** on a
100 MVA base: 1 p.u. of P is 100 MW, of Q 100 Mvar, and |V| = 1 p.u. is the
nominal voltage.

**Real and invented.** The equations are the real AC power flow. The line
impedances, the injection ranges and the set-points are invented, line charging
is neglected, and the generators' reactive-power limits are not enforced. Say so
whenever you quote a number from this notebook.

**The graph.** L5.1 describes a graph to a network by three arrays: the adjacency
**A**, the node features **X**, and the edge features **E**. Here they are for
one case.

In [ ]:
case = 0
print("A — 1 where a line joins two buses:")
print(A.astype(int))

print(f"\nX — what is given at each bus, case {case}. A 0 in P, Q or V_set means")
print("'not given, solved by the power flow'; the last three columns say which.\n")
print(core.error_table(
    [[i, core.BUS_TYPE[i]] + [f"{v:.3f}" for v in X[case, i]] for i in range(6)],
    ["bus", "type"] + data["feature_names"].tolist()))

print("\nE — one row per line: impedance z = R + jX and admittance y = 1/z\n")
print(core.line_table())

**What you should see.** The adjacency matrix from notebook 02; a table of X for
case 0 in which bus 0 has only `V_set = 1.030` and its flag, bus 1 has
`P = 0.476` and `V_set = 1.020`, and buses 2 to 5 have P and Q; and the table of
E, the eight lines with R, X and the admittance y = 1/(R + jX).

**X** holds only what is *given*. A quantity the power flow has to solve is
entered as 0, and the three type flags tell the model which zeros mean "not
given". The flags matter for a second reason: the model has no notion of bus
number, so "bus 0 is the reference" can only reach it as a feature, one that
moves with the bus when the buses are renumbered.

**E** describes each line by its impedance z = R + jX, or equivalently its
admittance y = 1/z. The AC power flow uses all of it. The graph network in this
notebook uses only A, so that its arithmetic stays that of notebook 02; giving
it E as well is a natural next step.

Now one case in full: what is injected at each bus, and the solved power flow.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.0, 4.4))
core.plot_graph(node_values=data["P"][case], ax=axes[0], annotate="{:+.2f}",
                cbar_label="P [p.u.]  (1 p.u. = 100 MW)",
                title=f"Case {case}: active power P into the network")
core.plot_graph(node_values=data["Q"][case], ax=axes[1], annotate="{:+.2f}",
                cbar_label="Q [p.u.]  (1 p.u. = 100 Mvar)",
                title=f"Case {case}: reactive power Q into the network")
plt.show()

print(f"Case {case} after the power flow is solved:\n")
print(core.bus_table(data["P"][case], data["Q"][case], Y[case, :, 1], Y[case, :, 0]))
print(f"\nP summed over all buses = losses in the lines: "
      f"{data['P'][case].sum():.4f} p.u. = {100 * data['P'][case].sum():.1f} MW")

**What you should see.** Two drawings of the network. The left colours each bus
by its active power P and writes the value beside it, in per unit on 100 MVA —
`+1.32` at the slack bus is 132 MW generated, `-0.84` at bus 3 is 84 MW drawn by
the load. The right does the same for the reactive power Q, in Mvar / 100. Red
is power into the network, blue is power taken out.

Then the case as a power-flow table, one row per bus:

| bus | type | P_gen | Q_gen | P_load | Q_load | \|V\| [p.u.] | θ [deg] |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | slack | 1.321 | 0.355 | 0 | 0 | 1.0300 | 0.00 |
| 1 | PV | 0.476 | 0.363 | 0 | 0 | 1.0200 | −1.70 |
| 2 | PQ | 0 | 0 | 0.774 | 0.169 | 1.0077 | −2.91 |
| 3 | PQ | 0 | 0 | 0.836 | 0.214 | 0.9921 | −4.37 |
| 4 | PQ | 0 | 0 | 0.662 | 0.144 | 0.9916 | −4.38 |
| 5 | PQ (HVDC) | 0.493 | 0 | 0 | 0.059 | 0.9917 | −3.08 |

and `losses in the lines: 0.0170 p.u. = 1.7 MW`.

Read it the way a power-system engineer would. The slack bus's 1.321 p.u. and
the PV bus's Q of 0.363 were not given: the power flow worked them out. The loads
take 2.272 p.u., buses 1 and 5 give 0.969, and the slack bus supplies the other
1.303 **plus** the 0.017 lost in the lines. The angles fall away from the slack
bus towards the loads, which is the direction active power flows in, and the
voltage sags below the generators' set-points at the load buses.

Now all 800 cases at once, per bus.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8.2, 6.4), sharex=True)
axes[0].boxplot([Y[:, b, 1] for b in range(6)], positions=range(6))
axes[0].axhline(0.95, color="#777777", ls="--", lw=1.1, label="limits 0.95 and 1.05 p.u.")
axes[0].axhline(1.05, color="#777777", ls="--", lw=1.1)
axes[0].set_ylabel("|V| [p.u.]")
axes[0].set_title("The 800 cases, per bus (box: the middle half of the cases)")
axes[0].legend(frameon=False, fontsize=8)
axes[1].boxplot([np.degrees(Y[:, b, 0]) for b in range(6)], positions=range(6))
axes[1].set_ylabel("angle theta [deg]")
axes[1].set_xticks(range(6))
axes[1].set_xticklabels([f"{b}\n{core.BUS_TYPE[b]}" for b in range(6)])
axes[1].set_xlabel("bus number")
for a in axes:
    a.grid(alpha=0.25)
fig.tight_layout()
plt.show()

print("per-bus standard deviation of theta [deg]:", np.round(np.degrees(Y[:, :, 0].std(axis=0)), 2))
print("per-bus standard deviation of |V| [p.u.]:  ", np.round(Y[:, :, 1].std(axis=0), 4))

**What you should see.** Two box plots with the bus number on the horizontal axis:
|V| in per unit on top, with dashed lines at the usual operating limits of 0.95
and 1.05 p.u., and θ in degrees below. The standard deviations over the 800
cases are

```
theta [deg]: [0.   0.76 0.8  1.36 1.36 1.47]
|V| [p.u.] : [0.     0.     0.0036 0.0069 0.0076 0.0086]
```

**Buses 0 and 1 do not move in |V|**: they are the slack and PV buses, and their
magnitude is a set-point, 1.03 and 1.02 p.u. in every case. **Bus 0 does not
move in θ either**: it is the reference. Everything else varies, the load buses
most, and every case stays inside 0.95–1.05 p.u.

**The two targets are on different scales.** The angle spreads over about
0.04 rad and |V| over about 0.02 p.u., around a mean of 1. A plain mean squared
error over both would weight them by those accidents of units. So each target is
shifted by its mean and divided by its standard deviation before training, and
converted back before reporting.

---

## 2 · Splitting, and scaling the targets

In [ ]:
n_train = 600
Y_mean, Y_scale = data["Y_mean"], data["Y_scale"]
Y_scaled = ((Y - Y_mean) / Y_scale).astype(np.float32)   # each target: mean 0, std 1

X_train, Y_train = X[:n_train], Y_scaled[:n_train]
X_test,  Y_test  = X[n_train:], Y_scaled[n_train:]
Y_test_physical  = Y[n_train:]                            # radians and per unit, for reporting

print("training cases :", X_train.shape[0])
print("held-out cases :", X_test.shape[0])
print("target mean  [theta rad, |V| p.u.]:", Y_mean)
print("target std   [theta rad, |V| p.u.]:", Y_scale)
print("scaled training targets, std:", np.round(Y_train.reshape(-1, 2).std(axis=0), 3))

**What you should see.** 600 training cases, 200 held out, target means
`[-0.0648  1.0004]` and standard deviations `[0.0413  0.0197]` — radians and per
unit — and scaled standard deviations `[1.001 1.002]`.

The split is a plain first-600 / last-200 cut — safe here because the cases are
independent draws, and not safe on the time series in notebook 04.

---

## 3 · The layer

This is notebook 02's message-passing layer, now in PyTorch:

$$\mathbf{H}^{(l+1)} = \tanh\!\left(
\mathbf{H}^{(l)}\mathbf{W}_{\mathrm{self}} + \mathbf{b}
\;+\;\mathbf{A}\,\mathbf{H}^{(l)}\mathbf{W}_{\mathrm{neigh}}\right)$$

Each bus keeps its own state through $\mathbf{W}_{\mathrm{self}}$ and adds up its
neighbours' states, $\mathbf{A}\mathbf{H}$, through $\mathbf{W}_{\mathrm{neigh}}$.
In L5.1's words: the message from a neighbour is a linear map of its state, the
messages are summed, and the sum is combined with the bus's own state and passed
through the activation. The same weights serve every bus.

Three layers are stacked, one per hop of the graph's diameter, and a final
linear map turns each bus's 32 numbers into its two predictions. The sizes and
the activation are set once, at the top of the cell. `tanh` rather than ReLU:
it is smooth and has a second derivative everywhere, which Part 2's physics
losses need (L4.1).

### Your turn

Write the layer and the network.

In [ ]:
# TODO 1 --- the message-passing layer and the network ------------------------------------------
# Two `...` to replace:
#   line 1  ->  ACTIVATION(self.self_lin(H) + A @ self.neigh_lin(H))   own state plus summed neighbours
#   line 2  ->  layer(A, H)                                             the layers one after the other

# The network's sizes and its activation, all in one place.
N_FEATURES = 6            # numbers given per bus: P, Q, V_set, is_slack, is_PV, is_PQ
HIDDEN     = 32           # numbers each bus carries from one layer to the next
N_TARGETS  = 2            # numbers predicted per bus: theta, |V|
N_LAYERS   = 3            # message-passing rounds; 3 = the graph's diameter (notebook 02)
ACTIVATION = torch.tanh   # smooth, with a second derivative everywhere (L4.1)


class GraphLayer(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.self_lin  = nn.Linear(n_in, n_out)                # a bus's own state
        self.neigh_lin = nn.Linear(n_in, n_out, bias=False)    # its neighbours' states

    def forward(self, A, H):                                   # H is (cases, 6, n_in)
        # A @ ... adds up the neighbours of every bus: the message passing
        return ...                     # <- ACTIVATION(self.self_lin(H) + A @ self.neigh_lin(H))


class GraphNet(nn.Module):
    def __init__(self, n_layers=N_LAYERS):
        super().__init__()
        sizes = [N_FEATURES] + [HIDDEN] * n_layers
        self.layers = nn.ModuleList([GraphLayer(sizes[k], sizes[k + 1])
                                     for k in range(n_layers)])
        self.head = nn.Linear(HIDDEN, N_TARGETS)               # per bus, no activation

    def forward(self, A, H):
        for layer in self.layers:
            H = ...                    # <- layer(A, H)
        return self.head(H)
# ------------------------------------------------------------------------------

In [ ]:
core.set_seed(0)
gnn = GraphNet()
n_gnn = core.count_parameters(gnn)

with torch.no_grad():
    probe = gnn(torch.tensor(A, dtype=torch.float32), torch.tensor(X_train[:4]))
print("shape check:", tuple(X_train[:4].shape), "->", tuple(probe.shape),
      "  (cases, buses, numbers per bus)")
print("parameters :", n_gnn)
for name, p in gnn.named_parameters():
    print(f"  {name:26s} {str(tuple(p.shape)):>10s}  {p.numel():5d}")

**What you should see.** `shape check: (4, 6, 6) -> (4, 6, 2)` — four cases,
six buses, six numbers in and two out per bus — and `parameters : 4642`,
listed layer by layer.

Six buses in, six buses out, and the number of buses appears nowhere in the
parameter list: every weight matrix is indexed by *features*, never by *buses*.
Add a seventh bus and this model runs unchanged.

---

## 4 · Training

Full batch — all 600 cases in every step — Adam, mean squared error on the
scaled targets, 1500 epochs. `core.train_graph` runs the loop; it is the same
four lines as always, with the adjacency passed through to the model.

### Your turn

In [ ]:
# TODO 2 --- train the graph network, then predict in physical units ---------------------------
# Two `...` to replace:
#   line 1  ->  core.train_graph(gnn, A, X_train, Y_train, epochs=1500, lr=0.01,
#                                X_val=X_test, Y_val=Y_test, verbose_every=300)
#   line 2  ->  out.numpy() * Y_scale + Y_mean                 back to radians and per unit
history_gnn = ...      # <- core.train_graph(gnn, A, X_train, Y_train, epochs=1500, lr=0.01, X_val=X_test, Y_val=Y_test, verbose_every=300)

def predict(model, A_any, X_any):
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(A_any, dtype=torch.float32), torch.tensor(X_any))
    return ...                           # <- out.numpy() * Y_scale + Y_mean

def rmse(pred, ref, ch):                 # ch 0 = angle [rad], ch 1 = |V| [p.u.]
    return float(np.sqrt(np.mean((pred[:, :, ch] - ref[:, :, ch]) ** 2)))

pred_gnn = predict(gnn, A, X_test)
rmse_theta_gnn = rmse(pred_gnn, Y_test_physical, 0)
rmse_volt_gnn  = rmse(pred_gnn, Y_test_physical, 1)
# ------------------------------------------------------------------------------

In [ ]:
print(f"\nangle RMSE: {rmse_theta_gnn:.5f} rad = {np.degrees(rmse_theta_gnn):.3f} degrees")
print(f"|V| RMSE  : {rmse_volt_gnn:.5f} p.u.")

train = history_gnn["train"]
jumps = int(np.sum(train[1:] > 2 * train[:-1]))
print(f"\nepochs at which the training loss more than doubled: {jumps} of {len(train)}")

core.plot_curves(history_gnn, title=f"Graph network: {N_LAYERS} layers, {n_gnn:,} parameters")
plt.show()

**What you should see.** A log every 300 epochs ending near `train 0.000284
val 0.000239`, then

```
angle RMSE: 0.00057 rad = 0.033 degrees
|V| RMSE  : 0.00033 p.u.

epochs at which the training loss more than doubled: 47 of 1500
```

and the two loss curves on top of each other, with sharp spikes.

In engineering units: three hundredths of a degree, and 0.03 % of nominal
voltage. The training and validation curves lie on top of each other, so the
network has not memorised the 600 cases.

**The spikes.** They are not noise in the data, and not mini-batch noise:
every step here uses all 600 cases, so the gradient is exact and the same
on every run. They come from the optimiser. Adam scales each weight's step by
its recent gradient size, so each step is about the learning rate, 0.01, however
small the gradient has become. Near the bottom of the loss, where the valley is
narrow, a step that size overshoots, the loss jumps by a factor of ten or more,
and it takes Adam some tens of epochs to shrink its step and settle lower than
before. A smaller learning rate gives fewer spikes — on our run `lr=0.003` halved
the count — at the price of slower progress early on. The final model is taken
at the last epoch, which here is between spikes.

**The results as a power-system engineer would show them.** Per bus, with the
bus number on the horizontal axis: the voltage profile on top, the angle profile
under it, the error at the bottom, and a table.

In [ ]:
case = 0                                  # the first held-out case
core.plot_power_flow(Y_test_physical[case, :, 1], Y_test_physical[case, :, 0],
                     {"graph network": (pred_gnn[case, :, 1], pred_gnn[case, :, 0])},
                     title=f"Held-out case {case}: reference power flow and graph network")
plt.show()

k = n_train + case                        # the same case in the full dataset
print(core.bus_table(data["P"][k], data["Q"][k], Y[k, :, 1], Y[k, :, 0],
                     extra={"|V| GNN [p.u.]": pred_gnn[case, :, 1],
                            "theta GNN [deg]": np.degrees(pred_gnn[case, :, 0])}))

**What you should see.** Three panels for held-out case 0, sharing the bus axis.

* **Voltage profile**, |V| in p.u., with the limits 0.95 and 1.05 p.u. dashed: the
  reference power flow (grey) and the graph network (blue) side by side at every
  bus, 1.03 and 1.02 at the two generators and just under 1 at the loads.
* **Angle profile**, θ in degrees: 0 at the slack bus, falling to about −7° at
  buses 3 and 4.
* **Error**, network minus reference: the angle error in degrees and the |V|
  error in per cent of 1 p.u. Both stay within ±0.12.

The table has the same case in the usual per-bus layout — P_gen, Q_gen, P_load,
Q_load, |V|, θ — with the graph network's |V| and θ in the last two columns: at
bus 4, 0.9888 p.u. and −7.10° from the power flow against 0.9880 p.u. and −7.14°
from the network.

---

## 5 · Accuracy against speed

The AC power flow is the ground truth here: every target in the dataset came
from it, and the graph network can only come close to it. So the graph network
is worth having only if it is **faster**. The question is by how much, and
from what size of grid.

Two ways of using a model matter in practice.

* **One case at a time** — an operator asks for one operating point.
* **Many cases at once** — screening hundreds of operating points or
  contingencies. The graph network then does all of them in one forward
  pass. Newton-Raphson cannot batch like that: every case has its own Jacobian
  and its own iterations, so it solves them one after another.

First, this six-bus grid, with the network you have just trained.

In [ ]:
# Time per case on this six-bus grid, both on the CPU. Each figure is the
# median of 7 runs after 2 warm-up runs (the first runs are slow for reasons
# that have nothing to do with the method: loading code, allocating memory).
P_given = X_test[:, :, 0].astype(float)          # the 200 held-out cases
Q_given = X_test[:, :, 1].astype(float)

def ac_all():                                    # Newton-Raphson, one case after another
    for k in range(len(X_test)):
        core.ac_power_flow(P_given[k], Q_given[k], tol=1e-8)

A_t = torch.tensor(A, dtype=torch.float32)
X_one, X_all = torch.tensor(X_test[:1]), torch.tensor(X_test)

def gnn_one():                                   # one case
    with torch.no_grad():
        gnn(A_t, X_one)

def gnn_all():                                   # all 200 cases in one forward pass
    with torch.no_grad():
        gnn(A_t, X_all)

t_ac          = core.median_time(ac_all, repeats=7) / len(X_test)
t_gnn_single  = core.median_time(gnn_one, repeats=7)
t_gnn_batched = core.median_time(gnn_all, repeats=7) / len(X_test)

print(core.error_table(
    [["AC power flow (Newton-Raphson)", f"{1e3 * t_ac:.4f}", "1"],
     ["graph network, one case", f"{1e3 * t_gnn_single:.4f}", f"{t_ac / t_gnn_single:.1f}"],
     ["graph network, 200 cases at once", f"{1e3 * t_gnn_batched:.4f}", f"{t_ac / t_gnn_batched:.0f}"]],
    ["method", "time per case [ms]", "times faster than the AC power flow"]))

**What you should see.** A table like

| method | time per case [ms] | times faster than the AC power flow |
| --- | --- | --- |
| AC power flow (Newton-Raphson) | 0.2531 | 1 |
| graph network, one case | 0.1163 | 2.2 |
| graph network, 200 cases at once | 0.0023 | 111 |

Timings depend on the machine and change from run to run — the batched figure
most, between about 70 and 150 times faster on our runs — so read the ratios,
not the digits. On six buses both take a fraction of a millisecond. One case at
a time, the graph network is only about twice as fast: at this size both times
are mostly Python overhead, not arithmetic. In a batch it is about a hundred
times faster, because one forward pass over 200 cases costs little more than a
pass over one.

Six buses is a toy. The real question is how the two times grow with the size
of the grid. Newton-Raphson solves a linear system with one unknown per bus (two
per PQ bus) at every iteration. A message-passing layer does a fixed amount of
work per bus and per line. Next, both are timed on larger grids.

**The two sides, stated plainly.** The AC power flow is Newton-Raphson with
dense NumPy matrices (`core.newton_raphson_dense`) and with `scipy.sparse`
matrices and a sparse LU solve (`core.newton_raphson`), whichever is faster at
that size — dense on small grids, sparse on large ones, as production programs
do. It starts from a flat start and stops at a mismatch below 1e-8 p.u. The
graph network is ours — three layers, 32 numbers per bus, tanh — with the
neighbour sum done over the list of lines, so its cost grows with the number of
lines. Both run on the CPU.

In [ ]:
# The same comparison on synthetic grids of N buses (core.synthetic_grid): a ring
# with chords, realistic line impedances, a slack bus, a PV generator every fourth
# bus, loads elsewhere. The AC power flow is solved for real on each grid. The
# graph network has the same layers as ours but random weights: how long a
# forward pass takes does not depend on the weights. About half a minute.
sweep = core.speed_sweep((6, 12, 24, 48, 96, 192, 384, 768))

for key, mode in (("t_gnn_single", "one case at a time"),
                  ("t_gnn_batched", "in a batch of 256"),
                  ("t_gnn_reach_batched", "in a batch, one layer per hop")):
    x = core.crossover(sweep, key)
    if x is None:
        print(f"graph network, {mode}: never faster than the AC power flow here")
    elif x <= sweep[0]["n"]:
        print(f"graph network, {mode}: faster at every size here, from {sweep[0]['n']} buses")
    else:
        print(f"graph network, {mode}: faster from about {x:.0f} buses")

core.plot_speed(sweep)
plt.show()

**What you should see.** One line per grid, from 6 to 768 buses; three lines
saying that the graph network is faster than the AC power flow at every size
here, in each of the three modes; and the plot, time per case against buses on
log-log axes. On our run:

| buses | AC power flow [ms] | GNN, one case [ms] | GNN, batch of 256 [ms per case] | GNN, batch, one layer per hop [ms per case] |
| --- | --- | --- | --- | --- |
| 6 | 0.154 | 0.133 | 0.0029 | 0.0020 |
| 48 | 0.413 | 0.406 | 0.0101 | 0.0277 |
| 96 | 1.694 | 0.224 | 0.0158 | 0.0665 |
| 768 | 14.503 | 0.735 | 0.2963 | 3.4805 |

**There is no crossover to find in a batch: the graph network is already about
fifty times faster at six buses**, and stays about fifty times faster up to 768.
**One case at a time the two are level on small grids** — the graph network is
1.0 to 1.5 times faster up to 48 buses, which is within the scatter of the
timing — and it pulls away from about a hundred buses, to about twenty times
faster at 768.

Three things keep this honest.

* **Both sides are Python.** At a few dozen buses both times are mostly
  overhead. A compiled power-flow program would solve six buses in far less
  than 0.2 ms, and against it the crossover would move up to larger grids. The
  trend with size is the robust result, not the small-grid numbers.
* **Depth has to grow with the grid.** Three layers let a bus hear from buses
  three lines away. A grid of 768 buses here has buses 33 lines from the slack
  bus, and a network that is to reach across it needs about that many layers.
  The purple curve pays for one layer per hop, and it is still faster than the
  AC power flow, by about four times at 768 buses — but the margin shrinks as
  the grid grows.
* **Speed is not accuracy.** These networks are untrained. Whether a graph
  network trained on a 768-bus grid reaches this notebook's accuracy is a
  separate question, and nothing here answers it. A fast answer that is wrong
  in the case that matters — the tripped line in section 9 — is worth nothing.

---

## 6 · The dense network

A dense network on the flattened 36 numbers, predicting the flattened 12 — the
model somebody would write who had never heard of graph networks. It is the
second learned model, and the one that shows what the graph structure buys.

### Your turn

In [ ]:
# TODO 3 --- the dense network -------------------------------------------------------------
# One `...` to replace:
#   line 1  ->  self.net(H.reshape(H.shape[0], -1)).reshape(H.shape[0], 6, N_TARGETS)
class DenseNet(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(6 * N_FEATURES, hidden), nn.Tanh(),
                                 nn.Linear(hidden, hidden), nn.Tanh(),
                                 nn.Linear(hidden, 6 * N_TARGETS))

    def forward(self, A, H):                      # A accepted and ignored
        # all 36 numbers of a case in one row in, 12 answers out, back to 6 buses x 2
        return ...     # <- self.net(H.reshape(H.shape[0], -1)).reshape(H.shape[0], 6, N_TARGETS)


core.set_seed(0)
mlp = DenseNet()
n_mlp = core.count_parameters(mlp)
history_mlp = core.train_graph(mlp, A, X_train, Y_train, epochs=1500, lr=0.01,
                               X_val=X_test, Y_val=Y_test)

pred_mlp = predict(mlp, A, X_test)
rmse_theta_mlp = rmse(pred_mlp, Y_test_physical, 0)
rmse_volt_mlp  = rmse(pred_mlp, Y_test_physical, 1)
# ------------------------------------------------------------------------------

In [ ]:
print(core.error_table(
    [["graph network", f"{n_gnn:,}", f"{rmse_theta_gnn:.5f}", f"{rmse_volt_gnn:.5f}"],
     ["dense network", f"{n_mlp:,}", f"{rmse_theta_mlp:.5f}", f"{rmse_volt_mlp:.5f}"]],
    ["model (error against the AC power flow)", "parameters", "angle RMSE [rad]",
     "|V| RMSE [p.u.]"]))

**What you should see.**

| model (error against the AC power flow) | parameters | angle RMSE [rad] | \|V\| RMSE [p.u.] |
| --- | --- | --- | --- |
| graph network | 4,642 | 0.00057 | 0.00033 |
| dense network | 7,308 | 0.00078 | 0.00041 |

The two networks are close, the graph network a little better with fewer
parameters. On thirty-six numbers and six hundred cases, a dense network is a
strong competitor. What the graph network buys is in the next three sections.

---

## 7 · Renumbering the buses

$$f(\mathbf{P}\mathbf{X},\; \mathbf{P}\mathbf{A}\mathbf{P}^\top)
= \mathbf{P}\, f(\mathbf{X}, \mathbf{A})$$

Notebook 02 showed this identity on one untrained layer. Here it is on the two
trained models: renumber the buses of every held-out case, ask again, and
compare.

In [ ]:
# The permutation test. Renumber the buses of every held-out case and ask both
# models again. np.einsum("ij,njf->nif", Pm, T) renumbers the bus axis of every case.
perm = [3, 0, 5, 2, 4, 1]                     # the bus now called 0 used to be called 3
Pm = core.permutation_matrix(perm)
relabel = lambda T: np.einsum("ij,njf->nif", Pm, T).astype(np.float32)
X_perm, A_perm, Y_perm = relabel(X_test), Pm @ A @ Pm.T, relabel(Y_test_physical)

gaps, rmse_perm = {}, {}
for name, model in (("graph network", gnn), ("dense network", mlp)):
    f_perm = predict(model, A_perm, X_perm)               # f(PX, P A P^T)
    p_f = relabel(predict(model, A, X_test))              # P f(X, A)
    gaps[name] = float(np.abs(f_perm[:, :, 0] - p_f[:, :, 0]).max())
    rmse_perm[name] = rmse(f_perm, Y_perm, 0)

gap_gnn, gap_mlp = gaps["graph network"], gaps["dense network"]
rmse_gnn_perm, rmse_mlp_perm = rmse_perm["graph network"], rmse_perm["dense network"]
print(core.error_table(
    [["graph network", f"{gap_gnn:.1e}", f"{rmse_theta_gnn:.5f}", f"{rmse_gnn_perm:.5f}"],
     ["dense network", f"{gap_mlp:.1e}", f"{rmse_theta_mlp:.5f}", f"{rmse_mlp_perm:.5f}"]],
    ["model", "largest |f(PX, PAP') - P f(X, A)| [rad]",
     "angle RMSE [rad], original numbering", "angle RMSE [rad], renumbered"]))

**What you should see.**

| model | largest \|f(PX, PAP') − P f(X, A)\| [rad] | angle RMSE, original numbering | angle RMSE, renumbered |
| --- | --- | --- | --- |
| graph network | 3.0e-08 | 0.00057 | 0.00057 |
| dense network | 1.3e-01 | 0.00078 | 0.04860 |

The graph network's answer follows the renumbering to within float32 rounding,
and its error is unchanged. The dense network's answer changes by up to 0.13 rad
(7°), and its error grows sixty-fold. Its first-layer weights are indexed by
position, so a renumbered input is simply a different input. The graph network
was never trained for this: it follows from the layer, in which no weight
belongs to a particular bus.

---

## 8 · How deep?

Notebook 02 measured this network's diameter: **three hops**. One
message-passing layer moves information one hop, so with fewer than three layers
the injection at bus 0 cannot reach bus 5 at all.

In [ ]:
# Depth: the same network with 1 to 6 message-passing layers. About two minutes.
depth_results = []          # (layers, parameters, angle RMSE [rad], |V| RMSE [p.u.])
for n_layers in (1, 2, 3, 4, 6):
    core.set_seed(0)
    model = GraphNet(n_layers)
    core.train_graph(model, A, X_train, Y_train, epochs=1500, lr=0.01)
    p = predict(model, A, X_test)
    depth_results.append((n_layers, core.count_parameters(model),
                          rmse(p, Y_test_physical, 0), rmse(p, Y_test_physical, 1)))

print(core.error_table([[d, f"{n:,}", f"{rt:.5f}", f"{rv:.5f}"]
                        for d, n, rt, rv in depth_results],
                       ["layers", "parameters", "angle RMSE [rad]", "|V| RMSE [p.u.]"]))

fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot([r[0] for r in depth_results], [r[2] for r in depth_results],
        "o-", lw=1.8, ms=8, color="#1f77b4", label="angle [rad]")
ax.plot([r[0] for r in depth_results], [r[3] for r in depth_results],
        "s-", lw=1.8, ms=8, color="#0f9d58", label="|V| [p.u.]")
ax.axvline(3, color="#999999", ls=":", lw=1.4, label="graph diameter, 3 hops")
ax.set_yscale("log"); ax.set_xticks([r[0] for r in depth_results])
ax.set_xlabel("message-passing layers")
ax.set_ylabel("held-out RMSE [rad or p.u.]")
ax.set_title("Held-out error against depth")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.**

| layers | parameters | angle RMSE [rad] | \|V\| RMSE [p.u.] |
| --- | --- | --- | --- |
| 1 | 482 | 0.00706 | 0.00139 |
| 2 | 2,562 | 0.00073 | 0.00031 |
| 3 | 4,642 | 0.00057 | 0.00033 |
| 4 | 6,722 | 0.00069 | 0.00028 |
| 6 | 10,882 | 0.00277 | 0.00191 |

and the same numbers plotted against depth, on a log scale.

One layer is ten times worse on angles than two. Two to four layers are about
equally good, three the best on angles. Six layers are worse again, by four to
seven times — even though they have the most parameters. That is
**over-smoothing** from notebook 02: each layer adds up neighbours, and after
enough rounds every bus's numbers look alike and the network can no longer tell
the buses apart. Depth on a graph buys reach, up to about the diameter, and
beyond it costs accuracy.

---

## 9 · A line trips

Line 3, between buses 1 and 3, is taken out of service: seven lines instead of
eight, and a new graph. The reference is a new AC power flow on the tripped
network, for 200 new operating points drawn the same way as before. Neither
network is retrained. The graph network is handed the new adjacency; the dense
network's input has no place for a topology, so it cannot be told.

### Your turn

In [ ]:
# TODO 4 --- trip line 1-3, no retraining --------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  predict(gnn, A_trip, trip["X"])      the graph network is handed the new A
#   line 2  ->  predict(mlp, A, trip["X"])           the dense network cannot be told
lines_trip = core.lines_without(3)               # line 3 is the one joining buses 1 and 3
trip = core.six_bus_dataset(n_cases=200, seed=77, lines=lines_trip)   # AC power flow, 7 lines
A_trip = trip["A"]

pred_gnn_trip = ...                   # <- predict(gnn, A_trip, trip["X"])
pred_mlp_trip = ...                   # <- predict(mlp, A, trip["X"])

rmse_theta_gnn_trip = rmse(pred_gnn_trip, trip["Y"], 0)
rmse_volt_gnn_trip  = rmse(pred_gnn_trip, trip["Y"], 1)
rmse_theta_mlp_trip = rmse(pred_mlp_trip, trip["Y"], 0)
rmse_volt_mlp_trip  = rmse(pred_mlp_trip, trip["Y"], 1)
# ------------------------------------------------------------------------------

In [ ]:
print(core.error_table(
    [["graph network (given the new A)", f"{rmse_theta_gnn:.5f}", f"{rmse_theta_gnn_trip:.5f}",
      f"{rmse_volt_gnn:.5f}", f"{rmse_volt_gnn_trip:.5f}"],
     ["dense network (cannot be told)", f"{rmse_theta_mlp:.5f}", f"{rmse_theta_mlp_trip:.5f}",
      f"{rmse_volt_mlp:.5f}", f"{rmse_volt_mlp_trip:.5f}"]],
    ["model", "angle RMSE [rad], intact", "angle RMSE [rad], tripped",
     "|V| RMSE [p.u.], intact", "|V| RMSE [p.u.], tripped"]))

case = 0
ref = trip["Y"][case]
core.plot_power_flow(ref[:, 1], ref[:, 0],
                     {"graph network, given the new A": (pred_gnn_trip[case, :, 1], pred_gnn_trip[case, :, 0]),
                      "dense network": (pred_mlp_trip[case, :, 1], pred_mlp_trip[case, :, 0])},
                     title=f"Line 1-3 tripped, case {case}: AC power flow and both networks")
plt.show()

**What you should see.**

| model | angle RMSE [rad], intact | angle RMSE [rad], tripped | \|V\| RMSE [p.u.], intact | \|V\| RMSE [p.u.], tripped |
| --- | --- | --- | --- | --- |
| graph network (given the new A) | 0.00057 | 0.05164 | 0.00033 | 0.04420 |
| dense network (cannot be told) | 0.00078 | 0.07767 | 0.00041 | 0.05536 |

and the three-panel plot for tripped case 0: the AC power flow (grey), the graph
network (blue) and the dense network (red).

**Look at the voltage profile first.** With line 1-3 out, bus 3 and bus 5 are fed
only through bus 4, and in the AC power flow the voltage at buses 3, 4 and 5
falls **below the 0.95 p.u. limit** — to about 0.91 at bus 3. That is the kind
of result an operator runs a contingency study to find. Both networks miss it:
they predict 0.96 to 0.99 p.u., inside the limits, and angles of −4° to −8° at
bus 3 where the power flow has −12°.

The graph network, given the new graph, is closer than the dense network at
buses 3, 4 and 5, where the errors are largest, and about a third better in
RMSE. But its error is now ninety times what it was on the intact network.
**Do not oversell it.** A network trained on one topology has learned that
topology's statistics along with the physics, and a new graph is outside
everything it has seen. Being able to accept the new graph is **necessary** for
transfer, not **sufficient**; training on the intact network plus a set of
tripped ones is what would make it sufficient — and the dense network has no way
to do that at any amount of data, because there is nowhere to put the topology.
Section 5's speed is worth having only once that is done: the AC power flow,
solved again on the new network, needs no retraining at all.

---

## 10 · Save

Notebook 05 reads this file.

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb03_gnn.npz")
np.savez(path,
         n_gnn=n_gnn, n_mlp=n_mlp,
         rmse_theta_gnn=rmse_theta_gnn, rmse_volt_gnn=rmse_volt_gnn,
         rmse_theta_mlp=rmse_theta_mlp, rmse_volt_mlp=rmse_volt_mlp,
         t_ac=t_ac, t_gnn_single=t_gnn_single, t_gnn_batched=t_gnn_batched,
         speed_sweep=np.array([[r["n"], r["t_nr"], r["t_gnn_single"], r["t_gnn_batched"],
                                r["t_gnn_reach_batched"]] for r in sweep]),
         gap_gnn=gap_gnn, gap_mlp=gap_mlp,
         rmse_gnn_perm=rmse_gnn_perm, rmse_mlp_perm=rmse_mlp_perm,
         rmse_theta_gnn_trip=rmse_theta_gnn_trip, rmse_volt_gnn_trip=rmse_volt_gnn_trip,
         rmse_theta_mlp_trip=rmse_theta_mlp_trip, rmse_volt_mlp_trip=rmse_volt_mlp_trip,
         depth_results=np.asarray(depth_results, dtype=float),
         val_gnn=history_gnn["val"], val_mlp=history_mlp["val"])
print("wrote", path)
core.saved(path)

**What you should see.** `wrote .../Ex05_outputs/nb03_gnn.npz`.

---

## 11 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. A power-flow bus is slack, PV or PQ. For each type, say which two of P, Q, |V| and theta are given and which two are solved, and explain why the slack bus's |V| and theta did not change across the 800 cases. Why must the bus type reach the graph network as a feature rather than as a bus number?
   *→ L5.1 Q6*
2. The graph network and the dense network reached similar held-out errors, but renumbering the buses made the dense network's angle error about sixty times larger and left the graph network's unchanged. Explain why, and say which of the two you would hand to a grid operator.
   *→ L5.1 Q9*
3. In the depth sweep the held-out error was largest with one layer and grew again with six. Explain both ends of the sweep using the graph's diameter and over-smoothing.
   *→ L5.1 Q8*
4. The graph network only approximates the AC power flow it was trained on, yet it answered many cases far faster, most of all in a batch. With line 1-3 tripped it missed voltages below 0.95 p.u. that the power flow found. When is the speed worth having, and what would have to be true before you used the network to screen contingencies?
   *→ L5.1 Q10*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.
